# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.  We will utilize the `lark` parsing library to define our language rules, generate a raw syntax tree, and then convert it into a nested tuple.

In [ ]:
from lark import Lark, Token

## Specification of the Grammar

Below is the grammar for our simple `C`-like language expressed as a `lark` *Extended Backus-Naur Form* (EBNF) string. 

Our programming language allows both single-line (`//`) and multi-line (`/* */`) comments.  We instruct the parser to entirely `%ignore` them alongside standard whitespace.

Note that we have to use a *raw string* here, because otherwise the escape sequence `\/` would not be allowed.

In [ ]:
sl_grammar = r"""
?start: program

program: stmnt* -> prog

?stmnt: "if" "(" bool_expr ")" stmnt       -> if_stmnt
      | "while" "(" bool_expr ")" stmnt    -> while_stmnt
      | "{" program "}"                    -> block
      | IDENTIFIER ":=" expr ";"           -> assign
      | expr ";"                           -> expr_stmnt

?bool_expr: expr "==" expr -> eq
          | expr "!=" expr -> ne
          | expr "<=" expr -> le
          | expr ">=" expr -> ge
          | expr "<" expr  -> lt
          | expr ">" expr  -> gt

?expr: expr "+" product -> add
     | expr "-" product -> sub
     | product

?product: product "*" factor -> mul
        | product "/" factor -> div
        | product "%" factor -> mod
        | factor

?factor: "(" expr ")"
       | NUMBER                       -> number
       | IDENTIFIER                   -> var
       | IDENTIFIER "(" expr_list ")" -> fct_call

expr_list: (expr ("," expr)*)? -> explist

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.INT   -> NUMBER
%import common.WS

// In order to show how it's done, I have defined the tokens
// CPP_COMMENT and C_COMMENT describing comments myself.  
CPP_COMMENT: /\/\/[^\n]*/
C_COMMENT:   "/*" /(.|\n)*?/ "*/"

// Ignore whitespace and comments
%ignore WS
%ignore C_COMMENT
%ignore CPP_COMMENT
"""

We instantiate the parser utilizing the *Look-Ahead Left-to-Right* (LALR) algorithm.

In [ ]:
parser = Lark(sl_grammar, parser='lalr')

## Generating the Abstract Syntax Tree (ADT)

To maintain a clean functional boundary, we manually walk the raw `lark.Tree` generated by the parser and map it strictly to a pure Python Abstract Data Type represented by nested tuples. This perfectly mirrors the structure needed by our evaluation and visualization functions.

`Symbol_Map` is a dictionary mapping grammar aliases to their ADT symbols.

In [ ]:
SYMBOL_MAP = {
    # Program and Statements
    'prog': '.',
    'if_stmnt': 'if',
    'while_stmnt': 'while',
    'expr_stmnt': 'expr',    
    # Boolean Operations
    'eq': '==', 'ne': '!=', 'le': '<=', 'ge': '>=', 'lt': '<', 'gt': '>',
    # Arithmetic Operations
    'add': '+', 'sub': '-', 'mul': '*', 'div': '/', 'mod': '%'
}

`ast_to_tuples` takes a node of the syntax tree and transforms it into a nested tuple.

In [ ]:
def ast_to_tuples(node):
    # Base case: Leaf node (Token)
    if isinstance(node, Token):
        if node.type == 'NUMBER':
            return int(node.value)
        return str(node.value)
    if node.data in SYMBOL_MAP:
        symbol = SYMBOL_MAP[node.data]
        return (symbol, *[ast_to_tuples(child) for child in node.children])
    match node.data:
        case 'block':
            return ast_to_tuples(node.children[0]) 
        case 'assign':
            return (':=', str(node.children[0]), ast_to_tuples(node.children[1]))
        case 'number': 
            return int(node.children[0])
        case 'var': 
            return str(node.children[0])
        case 'fct_call':
            f_name = str(node.children[0])
            args = ast_to_tuples(node.children[1])
            return ('call', f_name, *args)
        case 'explist':
            return tuple([ast_to_tuples(child) for child in node.children])
        case _:
            raise ValueError(f"Unknown syntax tree node: {node.data}")

The external helper file `AST2Dot.ipynb` is used to represent our generated nested tuple ADT graphically.

In [ ]:
%run AST2Dot.ipynb

The function `parse` takes a `file_name` as its sole argument. The file is read, passed through the `lark` parser, mapped to our nested tuple format, and visualized via `graphviz`.

In [ ]:
def parse(file_name):
    with open(file_name, 'r') as handle:
        program = handle.read()
    print(program)
    tree = parser.parse(program)  
    ast  = ast_to_tuples(tree)
    print(ast)
    return tuple2dot(ast)

In [ ]:
parse('sum.sl')

The function `execute_tuple` loops through statements linearly. The dictionary `Values` is the continuous state that gets updated as assignments occur.

In [ ]:
def execute_tuple(StatementList, Values={}):
    for stmnt in StatementList:
        execute(stmnt, Values)

The function `execute` matches recursively against the shape of the nested tuple statement node, triggering evaluation or recursive execution sequences.

In [ ]:
def execute(stmnt, Values):
    match stmnt:
        case ('.', *SL):
            execute_tuple(tuple(SL), Values)
        case (':=', var, value):
            Values[var] = evaluate(value, Values)
        case ('expr', expr):
            evaluate(expr, Values)
        case ('if', test, stmnt):
            if evaluate_bool(test, Values):
                execute(stmnt, Values)
        case ('while', test, stmnt):
            while evaluate_bool(test, Values):
                execute(stmnt, Values)
        case _:
            assert False, f'{stmnt} unexpected'

The function `evaluate_bool` executes conditional logical statements returning a pure boolean output.

In [ ]:
def evaluate_bool(expr: NestedTuple, Values: dict[str, Number]) -> bool:
    match expr:
        case ('==', lhs, rhs):
            return evaluate(lhs, Values) == evaluate(rhs, Values)
        case ('!=', lhs, rhs):
            return evaluate(lhs, Values) != evaluate(rhs, Values)
        case ('<=', lhs, rhs):
            return evaluate(lhs, Values) <= evaluate(rhs, Values)
        case ('>=', lhs, rhs):
            return evaluate(lhs, Values) >= evaluate(rhs, Values)
        case ('<', lhs, rhs):
            return evaluate(lhs, Values) <  evaluate(rhs, Values)
        case ('>', lhs, rhs):
            return evaluate(lhs, Values) >  evaluate(rhs, Values)
        case _:
            assert False, f'{expr} unexpected'

The function `evaluate` calculates specific numerical values given either primitives (variables, strings) or compound arithmetic operations. We utilize python's built-in `match/case` to guarantee exhaustive mapping of our ADT types.

In [ ]:
def evaluate(expr: NestedTuple, Values: dict[str, Number]) -> Number:
    match expr:
        case int():
            return expr
        case str():
            return Values[expr] 
        case ('call', 'read'):
            return int(input('Please enter a natural number: '))
        case ('call', 'print', expr):
            print(evaluate(expr, Values))
            return 0;
        case ('+', lhs, rhs):
            return evaluate(lhs, Values) + evaluate(rhs, Values)
        case ('-', lhs, rhs):
            return evaluate(lhs, Values) - evaluate(rhs, Values)
        case ('*', lhs, rhs):
            return evaluate(lhs, Values) * evaluate(rhs, Values)
        case ('/', lhs, rhs):
            return evaluate(lhs, Values) / evaluate(rhs, Values)
        case ('%', lhs, rhs):
            return evaluate(lhs, Values) % evaluate(rhs, Values)
        case _:
            assert False, f'{expr} unexpected'

In [ ]:
!cat sum.sl

Finally, we bring the whole pipeline together. `main` will open the program file, parse it fully using `lark`, build our structural tuples, and evaluate it under an isolated state dict.

In [ ]:
def main(file):
    with open(file, 'r') as handle:
        program = handle.read() 
    tree = parser.parse(program)
    ast  = ast_to_tuples(tree)
    display(tuple2dot(ast))
    Values = {}
    execute(ast, Values)

In [ ]:
main('sum.sl')

In [ ]:
!cat factorial.sl

In [ ]:
main('factorial.sl')